# 2026-10-13 pilot

**Author:** Kyle Shannon · **Notebook:** pilot · **Created:** 2026-10-13T01:55:10-07:00

**Purpose:** The pilot. Does an axis-of-travel representation emerge in a vanilla RNN trained only to predict the place cell population 0.5 s ahead? Simulated data only, a one-off study whose output is a go or no-go. This notebook is the single entry point: it holds almost no logic and calls `src/pilot/`. Sections 0 to 4 are here: the toy data and its sanity checks, training and the training checks, the direction and axis analysis, and the decision table. Figures come from `src/pilot/viz.py`.

In [ ]:
import os
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import subprocess
from datetime import date

from IPython.display import Image, display

import matplotlib.pyplot as plt
import numpy as np
import scienceplots  # noqa: F401  registers the SciencePlots styles

from pilot import viz, worlds
from pilot.config import CORRIDOR, PILOT, ROOM, SMOKE, WORLDS

plt.style.use(["science", "notebook", "no-latex"])

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip() or "unknown"
print(f"commit {commit}  {date.today()}")

## 0. Preset

`SMOKE` is short sessions and a couple of epochs, about two minutes for Run All. `PILOT` is the real run. Sessions are cached under `datasets/pilot/sessions/<preset>_<hash>/`: a file that exists is loaded, not recomputed. The hash is of every setting that changes a session (room, corridor, steering, cell settings, seeds, dt, session length), so a changed setting gets a new folder and old sessions are never reused by mistake. A change to the motion code is not a setting: delete the folders then.

In [ ]:
PRESET = {"SMOKE": SMOKE, "PILOT": PILOT}[os.environ.get("PILOT_PRESET", "SMOKE")]  # SMOKE by default; PILOT_PRESET=PILOT for the real run

cfg = PRESET
print(f"preset {cfg['name']}: {worlds.n_sessions(cfg)} sessions per world of {cfg['session_s']:.0f} s "
      f"({cfg['n_train']} train, {cfg['n_val']} val, {cfg['n_test']} test), dt {cfg['dt']} s")
print(f"room {ROOM[0]} m by {ROOM[1]} m, loop corridor {CORRIDOR} m wide; sessions in "
      f"{worlds.session_path(cfg, 'loop', 0).parent.relative_to(worlds.DATA_DIR.parent.parent)}")

## 1. Sessions and data sanity

Two worlds in the same 1 m by 1 m room, as in the brief: the open field, and the loop, which has a central hole that leaves a ring corridor 0.25 m wide. A corner of the ring is a 0.25 m square and belongs to no segment, so the four segments (bottom, top, left, right) run between the corners, each 0.5 m long. The layout of each world (its 100 place cell centres) is fixed by a layout seed, and the same 30 head direction cells are used in both.

The open field agent has stock RatInABox motion. The loop agent is steered: at every step its velocity is pulled (RatInABox `drift_velocity`, `drift_to_random_strength_ratio` 2) towards 0.15 m/s in the direction of the next corner of the corridor centre line, the target moves on when the agent is within 0.05 m of it, and after a random number of laps (mean 2) it turns round at a corner and goes the other way, so both directions are travelled. Its wall repulsion distance is a third of the corridor width. This is a deviation from the stock motion the brief specifies, made on purpose (issue 1 in the brief's Appendix C) and covered by the same rule 2 exception. Whether it also touches `AGENTS.md` rule 1 was raised as issue 19 in the brief's Appendix C: no input column carries a travel direction, but the paths are generated with a hand-coded one. The reading taken (issue 22) is that rule 1 is about what reaches training as an input or a label, so the steered paths are used; it stays flagged for the authors. The appendix at the end explains how the corridor and the steering strength were chosen, and why the room is the brief's square.

Everything is saved so that any input set can be built later: time, position, heading, speed, turn rate, place cell rates and head direction cell rates.

In [ ]:
layouts = worlds.make_layouts(cfg)
sessions = worlds.load_sessions(cfg, layouts)

for world in WORLDS:
    seeds = [s.seed for s in sessions[world]]
    splits = [worlds.split_of(cfg, i) for i in range(len(seeds))]
    print(f"{world}: {len(seeds)} sessions of {len(sessions[world][0].t)} steps, seeds {seeds[0]} to {seeds[-1]}, "
          f"train {splits.count('train')} val {splits.count('val')} test {splits.count('test')}")

### Per world: path, occupancy, speed, heading, turn rate, six place fields, three head direction tuning curves

The turn rate is the raw value RatInABox measures, so it has spikes where the agent bounces off a wall (up to pi / dt = 63 rad/s). The histogram is cut at 20 rad/s and the title gives the share beyond. Clipping is a later step, in the dataset.

**Figure 1.1: Open field, world sanity panel** (the title above the figure reads `open_field`).
Top row, left to right: the first 60 s of session 0; where the agent spent its time (all 20 sessions); speed; heading of moving steps; turn rate (log scale, clipped at ±20 rad/s); the 100 place-cell centres. Middle row: six place fields (cell indices spread evenly over 0 to 99; indices are sorted by y, so these sit at row ends along the right wall). Bottom row: tuning curves of head-direction cells 0, 10 and 20 of 30.

**Figure 1.2: Loop, world sanity panel** (title `loop`). The same panels for the steered loop. The grey box is the hole; place fields are cut where the hole's walls block the line of sight.

In [ ]:
for world in WORLDS:
    display(viz.world_figure(world, layouts[world], sessions[world], cfg))

### Checks

Each line is measured on all the loaded sessions of the preset. The limits are named values in `src/pilot/config.py`.

In [ ]:
for name, status, measured in worlds.check_sessions(cfg, layouts, sessions):
    print(f"[{status:>4}] {name}\n       {measured}")

**Heading on two axes in the loop.** With stock RatInABox motion in a 1 m square this check failed: 0.64 of the moving steps in a corridor segment ran along it, against the 0.8 limit (a uniform heading gives 0.5; the brief gives no number, the limit was chosen when the check was written and has not been changed). The loop agent is now steered along the corridor and the check passes, at 1.00 on the PILOT sessions.

Read that number for what it is. The steering pulls the agent along the corridor centre line, so 1.00 says the steering works, not that heading was measured in free behaviour. It also means the loop and the open field differ a lot in motion (heading on two axes against uniform), so the `place_motion` and `full` arms can tell the worlds apart from motion alone. That is by design, but it should be remembered when those arms are read.

The last section 1 check, "the section 8 measures on the raw place channels find no direction units and no axis units", needs `analysis.py`, so it runs in section 3 (the `inputs` rows).

**Null for the direction and axis tests** (issues 18 and 21 in the brief's Appendix C). The steered loop is nearly periodic (a lap, and a direction held for about two laps), so a circular time shift can put the segment and direction label back on top of its own unit and inflate the 99th percentile of the null. In the loop the tests therefore use a label permutation over segment passes, which does not depend on the loop being periodic. Circular shifts of at least 20 s stay only in the open field arena test, where the stock motion is not periodic. The shares quoted in issue 18 (a lap of about 35 s) were measured in the former 2.0 by 1.0 m room and have not been repeated in the square, where a lap takes about 20 s.


## 2. Training and training checks

Every arm learns the same task: from the noisy inputs of its input set, predict the clean place cell vector 10 steps (0.5 s) ahead. Only the input columns differ between input sets; every other setting is shared (`PILOT` and `SMOKE` in `src/pilot/config.py`). The arms:

- **trained**: a vanilla tanh RNN of 128 units, Adam, early stopping on the validation loss. Its diet is both worlds (every batch half open field, half loop) or the open field alone. The best-validation weights are kept.
- **untrained**: the same network with its input and recurrent weights frozen at their random draw. Only the readout is fitted, in closed form (ridge regression of the target on the hidden states of the training windows). This is what comes for free. The draw is PyTorch's default initialisation, which keeps the pre-activations small (sd about 0.15 on the SMOKE test sessions), so an untrained net works close to the linear range of tanh; and untrained seed s starts from the same weights as trained seed s. Both are open for the authors (issue 36 in the brief's Appendix C).
- **memoryless**: a one-hidden-layer network that sees the current step only.
- **persistence**: no model; the future place vector is predicted to equal the current one. Two versions are reported: the clean current vector (the stronger baseline, the one the check uses) and the noisy input as the network sees it.

`place_only` is the primary arm. `place_motion` and `full` need the `AGENTS.md` amendment (rule 1 for comparison arms), so the next cell runs them only when the text "comparison arm" is in `AGENTS.md`.

Runs are cached in `datasets/pilot/runs/<preset>_<data hash>/<arm>_s<seed>/` with `weights.pt`, `metrics.json` and `manifest.json` (seed, settings, git commit and whether the tree has changes, lockfile hash, torch version, status, times). A complete run is loaded, never retrained; a run folder made with other training settings stops the sweep rather than being reused. Stopping a cell (the interrupt button) saves the partial run as `interrupted`; it is kept, and the next run of the cell starts that arm again. Failed runs are kept too.

In [ ]:
import hashlib

from pilot import analysis, decision, study, train
from pilot.config import THRESHOLDS_SHA256
from proj_helpers.paths import repo_root

comparison = "comparison arm" in (repo_root() / "AGENTS.md").read_text()
INPUT_SETS = ("place_only", "place_motion", "full") if comparison else ("place_only",)
print(f'text "comparison arm" in AGENTS.md: {comparison}')
print("input sets that will run:", ", ".join(INPUT_SETS))
if not comparison:
    print("place_motion and full will not run until the amendment is in AGENTS.md")

jobs = train.sweep_jobs(cfg, INPUT_SETS)
print(f"{len(jobs)} runs:", ", ".join(f"{arm.name} x {len(train.seeds_of(arm, cfg))}"
                                      for input_set in INPUT_SETS for arm in train.arms_for(input_set)))

thr = analysis.load_thresholds()
sha = hashlib.sha256(analysis.THRESHOLDS.read_bytes()).hexdigest()
print(f"\nthresholds.yaml sha256 {sha}")
print(f"config THRESHOLDS_SHA256 {THRESHOLDS_SHA256}")
print("MATCH" if sha == THRESHOLDS_SHA256 else "MISMATCH: the pre-registered file has changed")

### Run the sweep

The order is, for each input set: the trained arm on both worlds, the memoryless net, the untrained draws, then the trained arms on the open field alone. The live probe (`viz.live_probe`) redraws every `LIVE_EVERY` epochs: the loss curves of the arm that is training, predicted against target place activity of four cells, and the PCA of the hidden state coloured by heading, all from one fixed validation window of the arm's diet pushed through the trace path. The memoryless net has no hidden state, so it shows the first two only. Complete runs are skipped, so this cell can be run again after an interrupt.

**Figure 2.1: Live training probe (last redraw).** Redrawn every 5 epochs during training on one fixed validation window: loss curves, predicted against target activity for a few place cells, and the hidden-state PCA coloured by heading.

In [ ]:
LIVE_EVERY = 1 if cfg["name"] == "smoke" else 5  # epochs between redraws of the live probe

live = viz.live_probe(cfg, sessions, thr, LIVE_EVERY)
results = train.run_sweep(cfg, sessions, INPUT_SETS, on_epoch=live)
print(f"{len(results)} of {len(jobs)} runs complete")

### Loss curves

Solid is the training loss and dashed the validation loss, one colour per seed; the dotted line is the test loss of clean persistence. The untrained net has no curve (its readout is not trained by steps). The training loss is the mean over an epoch of batches with a fresh noise draw each epoch, the validation loss is on fixed noise, so the two are not on exactly the same footing.

**Figure 2.2: Loss curves.** One panel per kind of trained net (trained on both worlds, memoryless, trained on the open field only), one line per seed. Solid is training loss, dashed is validation loss, log scale. The dotted horizontal line is the loss of clean persistence (copying the current place vector) at horizon 10.

In [ ]:
horizon = str(cfg["horizon"])
persistence = train.persistence_losses(cfg, sessions)
panels = [[(f"{input_set}: {train.label(arm)}",
            [(results[(arm, seed)]["metrics"]["train_loss"], results[(arm, seed)]["metrics"]["val_loss"])
             for seed in train.seeds_of(arm, cfg) if (arm, seed) in results])
           for arm in train.arms_for(input_set) if arm.kind != "untrained"] for input_set in INPUT_SETS]
display(viz.loss_figure(panels, persistence["clean"][horizon]["all"]))

### Test loss

Masked mean squared error on the test sessions of both worlds (and on each world alone), the inputs noisy as in training, mean and (sd) over the seeds of an arm. The networks are trained at the horizon of 10 steps. At the other horizons in the table the same network is scored against a target it was not trained for, so those columns show how a 0.5 s predictor behaves at 0.05 s and 1 s, not networks trained for those horizons; the checks below use horizon 10. Persistence does not depend on the input set, so its numbers repeat on every row.

In [ ]:
for world in ("all", *WORLDS):
    print(f"test loss, {world} test sessions")
    print("\n".join(train.loss_table(cfg, results, persistence, INPUT_SETS, world)))
    print()

### Linear decoders from the hidden state

A ridge regression from the hidden states of whole sessions (noisy inputs, the network started from rest at the start of each session) to position (x, y) and to heading (cos, sin), fitted on the validation sessions and scored by R-squared on the test sessions. Columns are all sessions pooled and each world on its own (fitted and scored within that world). Mean over seeds. Heading decodability in `place_only` and `place_motion` is a result in itself; it is not a check. The memoryless net has no hidden state to decode.

In [ ]:
print("\n".join(train.decoder_table(cfg, results, INPUT_SETS)))

### Training checks (brief 7.5)

The limits are from `thresholds.yaml`, written before any network was trained. The first check compares the worst trained seed with clean persistence, the stronger of the two baselines (the ratio against the noisy input is printed beside it). Nothing here was chosen after looking at an axis measure, and no axis measure has been computed.

In [ ]:
checks = train.section_2_checks(cfg, results, persistence, INPUT_SETS, thr)
for name, passed, numbers in checks:
    print(f"[{'PASS' if passed else 'FAIL'}] {name}\n       {numbers}")

failed, blockers = study.blockers_of(checks, results, jobs, sha)
reasons = failed + blockers
print()
print("Section 3 may run: all training checks pass, the sweep is complete and thresholds.yaml matches."
      if not reasons else "Section 3 may NOT run: " + "; ".join(reasons))

**Stop here and report before running section 3.** The training checks and the tables above are the numbers to report, with anything in the brief that turned out wrong or ambiguous.

## 3. Direction and axis analysis

For every run of the input sets above (the trained arms, and the untrained draws) the network is run over the whole test sessions of both worlds through the trace path, from rest at the start of each session, with the input noise of training. Each unit is min-max scaled over the pooled test steps of both worlds. Then, on the moving steps (brief section 8): the direction test (8.1) and the axis test (8.2) on the loop, with the pass-level label permutation null (issues 21 and 25), the arena test (8.3) on the open field with circular shifts, and the population measure (8.4). The same measures run on the noisy input channels as pseudo-units (kind `inputs`, from the trained-on-both-worlds net's input of seed 0; the input does not depend on the net). Every number is read against `src/pilot/thresholds.yaml`, which is hashed below and never edited.

The code is `src/pilot/study.py` (one row per run) and `src/pilot/decision.py` (section 4), both tested; this notebook only calls them.

The memoryless nets are not analysed: they have no recurrent hidden state and no trace path, and the decision table does not use them.

Section 3 refuses to run when a section 2 check failed (the brief's last row: fix the task first), unless `OVERRIDE_FAILED_CHECKS` is set. It always refuses when the sweep is not complete or `thresholds.yaml` does not match `THRESHOLDS_SHA256`. Results here are pilot observations under the rule 2 exception, not findings.

**SMOKE.** The SMOKE networks are 2-epoch junk. Sections 3 and 4 run on them only to show that the machinery works; none of their numbers says anything about the question.

In [ ]:
from pilot.config import FIGURES_DIR

# True lets section 3 run although a section 2 check failed. It is on for SMOKE only; set it by hand for PILOT.
OVERRIDE_FAILED_CHECKS = cfg["name"] == "smoke"

print(f"thresholds.yaml sha256 {sha}")
print(f"config.THRESHOLDS_SHA256 {THRESHOLDS_SHA256}  {'MATCH' if sha == THRESHOLDS_SHA256 else 'MISMATCH'}")
print(f"git: {study.thresholds_source()}")

study.check_gate(failed, blockers, OVERRIDE_FAILED_CHECKS)
if failed:
    print("OVERRIDE_FAILED_CHECKS is on: section 3 runs although the section 2 checks failed (" + "; ".join(failed) + ").")
if cfg["name"] == "smoke":
    print("SMOKE: 2-epoch networks. The machinery is shown to work; the numbers below say nothing about the question.")
print("input sets analysed:", ", ".join(INPUT_SETS))

In [ ]:
test_sessions = study.world_test_sessions(cfg, sessions)
rows, kept = study.analyse_runs(cfg, sessions, thr, study.analysis_jobs(cfg, INPUT_SETS))
print(f"{len(rows)} rows")

### Tables

One row per run: unit counts (and the share of all 128 units, or 100 place channels for `inputs`) that pass the direction test, the axis test, and of the axis units those that are released in the arena (do not persist); then the released units as a share of all units, which is the number the bar reads. `opposite / cross` and `parallel / cross` are the population measure of 8.4 (smaller is more axis-like). `inputs` rows are the noisy input channels as pseudo-units. Counts are over the test sessions only.

In [ ]:
print("\n".join(study.tables_text(rows, INPUT_SETS)))

### The section 1 check that waited for the analysis

The brief's section 1 asks that the section 8 measures on the raw place channels find no direction units and no axis units. It is read on the `place_only` input, which is place channels only. (In the other input sets the input also holds head direction channels, which are direction-tuned by construction; their `inputs` rows are reported above, not checked.)

In [ ]:
print(study.raw_place_check(rows))

### Released axis units against the bar

Each dot is one run: the released axis units as a percent of all units. The dotted line is `min_unit_fraction` and the dashed line the `untrained_percentile` of the untrained draws of that input set, both from `thresholds.yaml`.

**Figure 3.1: Released axis units against the bar.** For each trained seed, the share of all hidden units that are axis-tuned on the loop and released in the arena, against the 5 percent bar and the untrained reference.

In [ ]:
released = decision.released_fractions(rows, INPUT_SETS)
reference = decision.untrained_cutoffs(released, thr)
display(viz.released_figure(released, thr["min_unit_fraction"], reference))

### What the units do

For the `place_only` run of seed 0 trained on both worlds, the untrained draw of seed 0, and the input channels: the 8 numbers per unit that the tests read (units sorted by east-west selectivity, direction and axis units marked at the right), the opposite, parallel and cross distances of 8.4, mean activity by position on the loop and on the open field and by heading in each segment for the four units with the largest axis selectivity on the loop (chosen for display only; no threshold depends on them), and the hidden state PCA coloured by heading and by segment. Activity is scaled to [0, 1] per unit as for the tests.

The next cell draws eighteen figures, three groups of six, one group per network, in this order: **Figures 3.2a to 3.2f** the trained net (both worlds, seed 0), **3.3a to 3.3f** the untrained net (seed 0), **3.4a to 3.4f** the raw input channels (noisy place cells treated as units). A printed line above each group names the four units shown (the four with the largest east-west selectivity). Within a group:

- **a. Units by segment and direction.** The 8 mean activities per unit that the tests read (bottom, top, left, right, each east/west or north/south), units sorted most east-west first, with direction units and axis units marked.
- **b. Distances between the 8 mean hidden-state vectors.** The pairs behind the opposite/cross and parallel/cross ratios are outlined, with the three means.
- **c. Rate maps, loop.** Mean activity by position for the four units.
- **d. Rate maps, open field.** The same four units in the open field.
- **e. Heading tuning within each segment.** A row per unit, a column per segment, moving steps only.
- **f. Hidden-state PCA.** PC1 against PC2 and PC3 for moving steps in the loop, coloured by heading (top) and by segment (bottom).

In [ ]:
loop_pos, loop_heading, loop_speed = worlds.stack(test_sessions["loop"])
field_pos = worlds.stack(test_sessions["open_field"])[0]

for name in ("trained", "untrained", "inputs"):
    run = kept[name]
    pooled = {w: analysis.scale(np.concatenate(run["units"][w]), run["result"]["scaling"]) for w in WORLDS}
    h = pooled["loop"]
    means = analysis.cell_means(h, analysis.cell_masks(loop_pos, loop_heading, loop_speed, thr))
    units, selectivity = viz.most_axis_like(h, loop_pos, loop_heading, loop_speed, thr, 4)
    print(f"== place_only, {name}, seed 0; units shown {units.tolist()} (east-west selectivity "
          f"{np.round(selectivity, 2).tolist()})")
    for fig in (viz.cell_heatmap_figure(means, run["result"]["direction"].any(axis=1), run["result"]["axis"].any(axis=1)),
                viz.distance_figure(means),
                viz.rate_map_figure(h, loop_pos, units, "loop"),
                viz.rate_map_figure(pooled["open_field"], field_pos, units, "open_field"),
                viz.heading_tuning_figure(h, loop_pos, loop_heading, loop_speed, units, thr),
                viz.pca_figure(h, loop_pos, loop_heading, loop_speed, thr)):
        display(fig)

### One session streamed through the frozen network

The first 60 s of the first loop test session of the `place_only` run of seed 0 trained on both worlds, one frame every 4 steps (300 frames, 20 per second). Left: the rat on the map, coloured by heading. Middle: the hidden state on the first two principal components (fitted on all steps of the session), coloured by heading, with the whole stretch in grey. Right: the activity of the six units with the largest axis selectivity on that session (blue east-west, orange north-south). The gif is saved under `datasets/pilot/figures/`.

**Figure 3.5: Animation.** The first steps of one loop test session streamed through the frozen trained net (seed 0): the rat on the map, the hidden-state PCA point moving, and the most axis-like units (a display choice, not a test).

In [ ]:
session = test_sessions["loop"][0]
gif = viz.animate_session(FIGURES_DIR / f"{cfg['name']}_place_only_trained_both_s0_loop_test0.gif", session.pos,
                          session.heading, session.speed,
                          analysis.scale(kept["trained"]["units"]["loop"][0], kept["trained"]["result"]["scaling"]), thr)
print(gif.relative_to(repo_root()), f"{gif.stat().st_size / 1e6:.1f} MB")
display(Image(filename=str(gif)))

## 4. Decision table (brief section 9)

The numbers come from section 3 and the cutoffs from `thresholds.yaml`.

- **The bar** for an input set: every trained seed (trained on both worlds) has a released axis fraction of all units of at least `min_unit_fraction` and above the `untrained_percentile` of the untrained draws of the same input set (`seeds_required: all`).
- **Row 4**, trained matches untrained: no trained seed of any input set that ran is above the untrained cutoff.
- **Row 5**, axis units mostly persist: of the axis units of the trained seeds, pooled over seeds, more than `persist_majority` persist in the open field; read for every input set that has axis units.
- **Row 6**, same diet: the mean released fraction of the open-field-only diet is at least `diet_match_ratio` times the loop-and-arena one, for every input set that ran. When the loop-and-arena fraction is zero, the rule says they match.
- **Row 7**, direction without axis: summed over the trained `place_only` seeds, at least one direction unit and no axis unit.
- **Row 8**, the section 2 checks failed.
- The decision line is Go when row 1, 2 or 3 holds (the first that does), and No-go when none does. Rows 4 to 6 that hold are named beside it as qualifications, and row 7 is noted as an observation. This precedence is not in the brief. The rows and the line are in `src/pilot/decision.py` and are tested on fabricated rows for each outcome.

A row that needs an input set that is not being run is marked `n/a` and given no number. These are pilot observations under the rule 2 exception, not findings; the decision is the researcher's. On SMOKE every number is junk.

In [ ]:
print(f"thresholds.yaml sha256 {sha}  ({study.thresholds_source()})")
print(f"min_unit_fraction {thr['min_unit_fraction']}  untrained_percentile {thr['untrained_percentile']}  "
      f"seeds_required {thr['seeds_required']}  persist_majority {thr['persist_majority']}  "
      f"diet_match_ratio {thr['diet_match_ratio']}")
if cfg["name"] == "smoke":
    print("SMOKE: 2-epoch networks and one untrained draw; every number below is junk and only shows the machinery.")
print()

facts = decision.facts_of(rows, INPUT_SETS, thr)
table = decision.table_of(facts, INPUT_SETS, thr, failed)
print("\n".join(decision.table_lines(table)))

verdict = decision.verdict_of(table, failed)
if cfg["name"] == "smoke":
    verdict = "SMOKE, meaningless: " + verdict
print(f"\nDecision: {verdict}")
print(f"thresholds.yaml sha256 {sha}  ({study.thresholds_source()})")
print("Pilot observations under the rule 2 exception, not findings; the decision is the researcher's.")

## Appendix. How the loop geometry was chosen

**Why.** With stock RatInABox motion in a 1 m square with a 0.25 m corridor, 0.64 of the moving steps in a corridor segment ran along it (limit 0.8, uniform 0.5). The heading decorrelates in about 1.3 s, which is about 0.16 m of travel, too short for a 0.25 m corridor to line it up. Three levers were available: a narrower corridor, a longer room, and steering the agent along the corridor. Instead of choosing by eye, `src/pilot/sweep.py` ran a grid with a selection rule written down before the first run. The rule is about data quality only (heading, stalls, speed, reversals, coverage of the eight segment and direction cells). No axis measure was looked at, and none exists yet.

**Sweep.** The agent alone, no cells. Five sessions of 600 s per row, seeds 5000 to 5004 (used nowhere else, the same in every row). Rooms 1.0 by 1.0 m (the reference) and 2.0 by 1.0 m. Corridor widths 0.25 (the stock reference), 0.15, 0.10 and 0.076 m. Steered rows: `drift_to_random_strength_ratio` 2, 5 and 10, with `wall_repel_distance` a third of the width. Also run for comparison: the stock rows (no steering, RatInABox's default repel distance 0.1 m) and the widest corridor with the default 0.1 m repel distance. Steering: the velocity is pulled towards 0.15 m/s in the direction of the next corner of the corridor centre line; the target moves on within 0.05 m of the corner; after a geometric number of laps (mean 2) the agent turns round at a corner. Those three numbers were fixed before the sweep and not swept. The open field keeps stock motion. Run once, about 10 minutes: `pixi run python -m pilot.sweep`.

**Rule.** A steered row passes when all of these hold (the stock and comparison rows are printed but never chosen):

- a. Of the moving steps (above 0.03 m/s) inside corridor segments, corners excluded, at least 0.80 head within 45 degrees of the corridor direction.
- b. The longest time below 0.02 m/s is at most 5 s.
- c. The mean speed is between 0.12 and 0.18 m/s.
- d. Fewer than 2 percent of turn rates are at the reversal cap, read as |turn rate| above 0.9 pi / dt (a full reversal in one step is pi / dt).
- e. In at least 80 percent of the sessions every one of the 8 segment and direction cells has at least 5 s.

Among the passing rows: the widest corridor, then the lowest drift ratio, then the 2.0 by 1.0 room (the rule as written before the run; see the result for what became of the last tie-break). If nothing had passed, nothing would have been relaxed. The two best rows would have been reported and the old geometry kept.

**Table.** A star marks a criterion that is not met. Bad steps are positions in the hole or outside the room. The last column is the number of criteria met, or PASS.

| row | box (m) | corridor (m) | repel (m) | ratio | a along | b stall (s) | c speed (m/s) | d reversed (%) | e sessions | bad steps | met |
|---|---|---|---|---|---|---|---|---|---|---|---|
| stock | 1.0x1.0 | 0.25 | 0.1 default | - | 0.65\* | 0.7 | 0.125 | 0.1 | 5/5 | 0 | 4/5 |
| stock | 1.0x1.0 | 0.15 | 0.1 default | - | 0.74\* | 1.2 | 0.112\* | 0.2 | 5/5 | 0 | 3/5 |
| stock | 1.0x1.0 | 0.1 | 0.1 default | - | 0.84 | 0.9 | 0.100\* | 0.3 | 5/5 | 0 | 4/5 |
| stock | 1.0x1.0 | 0.076 | 0.1 default | - | 0.87 | 1.2 | 0.096\* | 0.4 | 5/5 | 0 | 4/5 |
| stock | 2.0x1.0 | 0.25 | 0.1 default | - | 0.65\* | 0.7 | 0.122 | 0.1 | 5/5 | 0 | 4/5 |
| stock | 2.0x1.0 | 0.15 | 0.1 default | - | 0.75\* | 0.9 | 0.113\* | 0.2 | 4/5 | 0 | 3/5 |
| stock | 2.0x1.0 | 0.1 | 0.1 default | - | 0.84 | 1.7 | 0.103\* | 0.2 | 5/5 | 0 | 4/5 |
| stock | 2.0x1.0 | 0.076 | 0.1 default | - | 0.86 | 1.2 | 0.097\* | 0.3 | 5/5 | 0 | 4/5 |
| steered | 1.0x1.0 | 0.25 | 0.083 | 2 | 1.00 | 0.2 | 0.143 | 0.0 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.25 | 0.083 | 5 | 1.00 | 0.1 | 0.147 | 0.0 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.25 | 0.083 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.15 | 0.050 | 2 | 1.00 | 0.2 | 0.143 | 0.0 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.15 | 0.050 | 5 | 1.00 | 0.1 | 0.147 | 0.0 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.15 | 0.050 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.1 | 0.033 | 2 | 1.00 | 0.2 | 0.143 | 0.0 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.1 | 0.033 | 5 | 1.00 | 0.1 | 0.147 | 0.0 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.1 | 0.033 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.076 | 0.025 | 2 | 0.99 | 1.0 | 0.137 | 0.1 | 5/5 | 0 | PASS |
| steered | 1.0x1.0 | 0.076 | 0.025 | 5 | 0.79\* | 1.4 | 0.039\* | 5.4\* | 2/5\* | 0 | 1/5 |
| steered | 1.0x1.0 | 0.076 | 0.025 | 10 | 0.70\* | 6.4\* | 0.011\* | 14.8\* | 0/5\* | 0 | 0/5 |
| steered | 2.0x1.0 | 0.25 | 0.083 | 2 | 1.00 | 0.1 | 0.144 | 0.0 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.25 | 0.083 | 5 | 1.00 | 0.1 | 0.148 | 0.0 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.25 | 0.083 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.15 | 0.050 | 2 | 1.00 | 0.2 | 0.144 | 0.0 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.15 | 0.050 | 5 | 1.00 | 0.1 | 0.148 | 0.0 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.15 | 0.050 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.1 | 0.033 | 2 | 1.00 | 0.1 | 0.144 | 0.0 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.1 | 0.033 | 5 | 1.00 | 0.1 | 0.148 | 0.0 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.1 | 0.033 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.076 | 0.025 | 2 | 0.99 | 1.0 | 0.132 | 0.3 | 5/5 | 0 | PASS |
| steered | 2.0x1.0 | 0.076 | 0.025 | 5 | 0.69\* | 1.1 | 0.026\* | 6.1\* | 0/5\* | 0 | 1/5 |
| steered | 2.0x1.0 | 0.076 | 0.025 | 10 | 0.69\* | 6.4\* | 0.010\* | 14.7\* | 0/5\* | 0 | 0/5 |
| repel 0.1 | 1.0x1.0 | 0.25 | 0.100 | 2 | 1.00 | 0.1 | 0.143 | 0.0 | 5/5 | 0 | PASS |
| repel 0.1 | 1.0x1.0 | 0.25 | 0.100 | 5 | 1.00 | 0.1 | 0.147 | 0.0 | 5/5 | 0 | PASS |
| repel 0.1 | 1.0x1.0 | 0.25 | 0.100 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |
| repel 0.1 | 2.0x1.0 | 0.25 | 0.100 | 2 | 1.00 | 0.1 | 0.144 | 0.0 | 5/5 | 0 | PASS |
| repel 0.1 | 2.0x1.0 | 0.25 | 0.100 | 5 | 1.00 | 0.1 | 0.148 | 0.0 | 5/5 | 0 | PASS |
| repel 0.1 | 2.0x1.0 | 0.25 | 0.100 | 10 | 1.00 | 0.0 | 0.149 | 0.1 | 5/5 | 0 | PASS |

**Result.** Every steered row passes except the 0.076 m corridor at ratios 5 and 10, where the agent barely moves (mean speed 0.01 to 0.04 m/s, 5 to 15 percent of turn rates at the reversal cap; a check after the sweep found the ratio 10 agent sitting at one corner for most of a session). No row had a position in the hole or outside the room. The rule's tie-breaks therefore picked the widest corridor (0.25 m), the lowest ratio (2) and the 2.0 by 1.0 room (`sweep.choose` still returns it, since it records the rule as written). The last tie-break was then reverted, for the reasons below. **Used: 1.0 by 1.0 m room for both worlds, 0.25 m corridor, drift ratio 2, repel distance 0.083 m.**

**What the table does and does not say.**

- The criteria do not separate the passing rows. Criterion a is 1.00 in every one of them because the steering sets the heading; the choice rests on the order of the tie-breaks, not on a measured advantage of one passing row over another.
- Stock motion in the 0.25 m corridor gives 0.65 in both rooms, as the Stage 1 sessions did. Narrowing alone lifts it to 0.84 to 0.87 at 0.10 and 0.076 m, but slows the agent to 0.096 to 0.103 m/s (criterion c fails), and stock rows are not eligible.
- The default repel distance (0.1 m) gives essentially the same results as a third of the width in the 0.25 m corridor, so the repel distance is not what makes it pass.
- The room did no measurable work. Stock motion gives 0.65 in the 1.0 by 1.0 m room and in the 2.0 by 1.0 m room, and the steered 1.0 by 1.0 m row at 0.25 m with ratio 2 passes with essentially the same numbers as the 2.0 by 1.0 m row (along 1.00 in both, stall 0.2 s against 0.1 s, speed 0.143 against 0.144 m/s, 0.0 percent reversed, 5 of 5 sessions). Steering did the work. The 2.0 by 1.0 m room came from the last tie-break alone, which prefers the box that departs from the brief.
- The 2.0 by 1.0 m room was lopsided, which is why it was dropped. Measured on the 5 sweep sessions, each of the four east-west cells held 13 to 17 percent of session time and each of the four north-south cells 4 to 6 percent (14 to 16 and 5 percent on the 20 PILOT loop sessions of that room). In the 1.0 by 1.0 m room all eight cells hold 8 to 9 percent of session time (8.0 to 8.7 percent pooled on the 20 PILOT loop sessions). The shortest cell in one session is 38 s there on the sweep sessions against 21 s in the 2.0 by 1.0 m room, and 24 s on the PILOT sessions against 11 s. The left and right segments of the long room were 0.5 m long, the top and bottom 1.5 m; its open field was not isotropic (issue 20 in the brief's Appendix C), and the 100 place cells in the open field covered 2 m2 instead of 1 m2.
- Reverting to the brief's square is one line (`ROOM = (1.0, 1.0)` in `config.py`). The session folder hash changed, so the sessions were regenerated; the cached ones are 1.0 by 1.0 m. Going back to the rectangle is the same line (issues 17 and 20 in the brief's Appendix C).
- Steering is a deviation from stock motion, and so from rule 2 of the brief (behavior validated against real behavior). It falls under the same exception as the stock motion did.
